In [1]:
import json
import numpy as np
import pandas as pd
import os
import ast
import gc
import torch
import itertools
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util, CrossEncoder
from FlagEmbedding import BGEM3FlagModel

# ==========================================
# 0. 設定與資料準備
# ==========================================
DATA_PATH = '../data/train_QA.csv'
INDEX_PATH = '../index.json'
OUTPUT_DIR = 'ultimate_benchmark_results'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

print(">>> Loading Data...")
with open(INDEX_PATH, 'r', encoding='utf-8') as f:
    index_data = json.load(f)
    
documents_pure = [item['text'] for item in index_data]
doc_ids_map = [str(item['id']) for item in index_data] 

train_df = pd.read_csv(DATA_PATH)
questions_pure = train_df['question'].to_list()
q_ids = train_df['id'].to_list()

# 處理 Ground Truth
ref_ids = train_df['ref_id'].to_list()
ref_ids = ["['" + e + "']" if e[0]!='[' else e for e in ref_ids]
ref_ids = [ast.literal_eval(e.replace('"', '').replace("'", '"')) if isinstance(e, str) and '[' in e else [e] for e in ref_ids]

def cleanup():
    """強制回收記憶體，防止 OOM"""
    gc.collect()
    torch.cuda.empty_cache()

def calculate_recall(hits_list, k_list=[1, 5, 10, 20, 50]):
    """計算 Recall@K"""
    metrics = {k: 0 for k in k_list}
    total = 0
    for i, hits in enumerate(hits_list):
        target_ids = set(ref_ids[i])
        if "is_blank" in target_ids: continue
        total += 1
        # 取出 doc_id 並去除後綴 (如 doc_1_crop -> doc_1)
        retrieved = [doc_ids_map[h['corpus_id']].split('_')[0] for h in hits]
        for k in k_list:
            if (target_ids & set(retrieved[:k])) == target_ids:
                metrics[k] += 1
    return {f"Recall@{k}": v/total for k, v in metrics.items()}

# 用於儲存第一階段所有結果 (Candidate Pool)
candidates_cache = {} 

# # ==========================================
# # 1. 第一階段：Retrieval (各取 Top-100)
# # ==========================================
# print("\n" + "="*40)
# print("PHASE 1: Retrieval Candidate Generation")
# print("="*40)

# # --- 1. Sparse: TF-IDF ---
# print("[1/6] Running Sparse: TF-IDF...")
# tfidf = TfidfVectorizer(stop_words='english')
# tfidf_doc = tfidf.fit_transform(documents_pure)
# tfidf_q = tfidf.transform(questions_pure)
# sims = cosine_similarity(tfidf_q, tfidf_doc)
# hits = []
# for s in sims:
#     top_idx = np.argsort(s)[::-1][:100]
#     hits.append([{'corpus_id': i, 'score': float(s[i])} for i in top_idx])
# candidates_cache['TF-IDF'] = hits
# del tfidf, tfidf_doc, tfidf_q, sims
# cleanup()

# --- 2. Sparse: BM25 ---
print("[2/6] Running Sparse: BM25...")
tokenized_corpus = [doc.lower().split() for doc in documents_pure]
bm25 = BM25Okapi(tokenized_corpus)
hits = []
for q in questions_pure:
    s = bm25.get_scores(q.lower().split())
    top_idx = np.argsort(s)[::-1][:100]
    hits.append([{'corpus_id': i, 'score': float(s[i])} for i in top_idx])
candidates_cache['BM25'] = hits
del bm25, tokenized_corpus
cleanup()

# # --- 3. Sparse: BGE-M3 (Learned Sparse) ---
# print("[3/6] Running Sparse: BGE-M3 Sparse...")
# # 注意: BGE-M3 比較吃顯存，跑完立刻釋放
# model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)
# d_sparse = model.encode(documents_pure, return_dense=False, return_sparse=True, return_colbert_vecs=False)['lexical_weights']
# q_sparse = model.encode(questions_pure, return_dense=False, return_sparse=True, return_colbert_vecs=False)['lexical_weights']
# hits = []
# for q_lex in tqdm(q_sparse, desc="BGE Sparse"):
#     scores = [model.compute_lexical_matching_score(q_lex, d_lex) for d_lex in d_sparse]
#     top_idx = np.argsort(scores)[::-1][:100]
#     hits.append([{'corpus_id': i, 'score': float(scores[i])} for i in top_idx])
# candidates_cache['BGE-M3-Sparse'] = hits
# # 保留 model 給下一步 Dense 用，只刪除 sparse data
# del d_sparse, q_sparse 
# cleanup()

# # --- 4. Dense: BGE-M3 (Dense) ---
# print("[4/6] Running Dense: BGE-M3 Dense...")
# # 直接使用剛剛載入的 model
# d_dense = model.encode(documents_pure, return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
# q_dense = model.encode(questions_pure, return_dense=True, return_sparse=False, return_colbert_vecs=False)['dense_vecs']
# raw_hits = util.semantic_search(q_dense, d_dense, top_k=100)
# candidates_cache['BGE-M3-Dense'] = raw_hits
# del model, d_dense, q_dense
# cleanup()

# # --- 5. Dense: Nomic v1.5 ---
# print("[5/6] Running Dense: Nomic v1.5...")
# model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', trust_remote_code=True)
# # Nomic 建議加 prefix
# d_emb = model.encode(["search_document: " + d for d in documents_pure], show_progress_bar=True)
# q_emb = model.encode(["search_query: " + q for q in questions_pure])
# raw_hits = util.semantic_search(q_emb, d_emb, top_k=100)
# candidates_cache['Nomic-v1.5'] = raw_hits
# del model, d_emb, q_emb
# cleanup()

# # --- 6. Dense: MiniLM-L6 ---
# print("[6/6] Running Dense: all-MiniLM-L6-v2...")
# model = SentenceTransformer('all-MiniLM-L6-v2')
# d_emb = model.encode(documents_pure, show_progress_bar=True)
# q_emb = model.encode(questions_pure)
# raw_hits = util.semantic_search(q_emb, d_emb, top_k=100)
# candidates_cache['MiniLM-L6'] = raw_hits
# del model, d_emb, q_emb
# cleanup()

# --- 輸出第一階段 Recall ---
print("\n>>> Calculating Stage 1 Recall...")
stage1_res = []
for name, hits in candidates_cache.items():
    metrics = calculate_recall(hits)
    metrics['Model'] = name
    stage1_res.append(metrics)
    print(f"  {name}: Recall@5={metrics['Recall@5']:.2%}, Recall@50={metrics['Recall@50']:.2%}")

# pd.DataFrame(stage1_res).to_csv(os.path.join(OUTPUT_DIR, 'stage1_recall_results.csv'), index=False)


>>> Loading Data...
[2/6] Running Sparse: BM25...

>>> Calculating Stage 1 Recall...
  BM25: Recall@5=89.74%, Recall@50=100.00%


In [2]:
# ==========================================
# 2. 第二階段：Reranking (Top-50 -> Top-5)
# ==========================================
print("\n" + "="*40)
print("PHASE 2: Reranking Evaluation")
print("="*40)

rerank_results = []

# 定義 Reranker 清單 (保證穩跑版)
rerankers = [
    # 1. Cross-Encoder (最準)
    # ('Cross', 'cross-encoder/ms-marco-MiniLM-L-6-v2'),
    # ('Cross', 'BAAI/bge-reranker-v2-m3'),
    # ('Cross', 'jinaai/jina-reranker-v1-turbo-en'), # Jina Reranker 歸類為 Cross
    
    # 2. Bi-Encoder (最快，作為 Reranker 用)
    ('Bi', 'jinaai/jina-embeddings-v2-base-en'),
    ('Bi', 'Alibaba-NLP/gte-large-en-v1.5'),
    
    # 3. Late Interaction (BGE-M3 ColBERT Mode)
    ('Late', 'BAAI/bge-m3')
]

for r_type, r_name in rerankers:
    print(f"\nProcessing Reranker: [{r_type}] {r_name}")
    
    model = None
    try:
        # 根據類型載入模型
        if r_type == 'Cross':
            model = CrossEncoder(r_name, trust_remote_code=True)
        elif r_type == 'Bi':
            model = SentenceTransformer(r_name, trust_remote_code=True)
        elif r_type == 'Late':
            model = BGEM3FlagModel(r_name, use_fp16=True)
            
        # 對每個 Retriever 的結果進行 Rerank
        for ret_name, initial_hits in candidates_cache.items():
            if ret_name != 'BM25':
                continue
            # 取前 50 名
            top_50_hits = [h[:50] for h in initial_hits]
            reranked_hits = []
            
            # --- Cross-Encoder Logic ---
            if r_type == 'Cross':
                pairs = []
                counts = []
                for i, hits in enumerate(top_50_hits):
                    q = questions_pure[i]
                    p = [[q, documents_pure[h['corpus_id']]] for h in hits]
                    pairs.extend(p)
                    counts.append(len(p))
                
                # Batch Predict
                scores = model.predict(pairs, batch_size=8, show_progress_bar=False)
                
                cursor = 0
                for i, c in enumerate(counts):
                    s = scores[cursor:cursor+c]
                    new_h = [{'corpus_id': h['corpus_id'], 'score': float(s[j])} for j, h in enumerate(top_50_hits[i])]
                    reranked_hits.append(sorted(new_h, key=lambda x: x['score'], reverse=True))
                    cursor += c

            # --- Bi-Encoder Logic ---
            elif r_type == 'Bi':
                for i, hits in enumerate(tqdm(top_50_hits, desc=f"Reranking {ret_name}", leave=False)):
                    q = questions_pure[i]
                    docs = [documents_pure[h['corpus_id']] for h in hits]
                    
                    q_emb = model.encode(q)
                    d_emb = model.encode(docs, batch_size=2)
                    scores = util.cos_sim(q_emb, d_emb).numpy()[0]
                    
                    new_h = [{'corpus_id': h['corpus_id'], 'score': float(scores[j])} for j, h in enumerate(hits)]
                    reranked_hits.append(sorted(new_h, key=lambda x: x['score'], reverse=True))
                    
            # --- Late Interaction (BGE-M3) Logic ---
            elif r_type == 'Late':
                # 這裡只處理 BGE-M3，其他的 Late Interaction 略過以求穩定
                for i, hits in enumerate(tqdm(top_50_hits, desc=f"Reranking {ret_name}", leave=False)):
                    q = questions_pure[i]
                    docs = [documents_pure[h['corpus_id']] for h in hits]
                    
                    q_vec = model.encode(q, return_colbert_vecs=True, return_dense=False, return_sparse=False)['colbert_vecs'][0]
                    d_vecs = model.encode(docs, batch_size=4, return_colbert_vecs=True, return_dense=False, return_sparse=False)['colbert_vecs']
                    
                    scores = [model.colbert_score(q_vec, d) for d in d_vecs]
                    
                    new_h = [{'corpus_id': h['corpus_id'], 'score': float(scores[j])} for j, h in enumerate(hits)]
                    reranked_hits.append(sorted(new_h, key=lambda x: x['score'], reverse=True))

            # 計算並記錄結果
            metrics = calculate_recall(reranked_hits, k_list=[1, 5])
            rerank_results.append({
                'Retriever': ret_name,
                'Reranker_Type': r_type,
                'Reranker_Name': r_name,
                'Final_Recall@1': metrics['Recall@1'],
                'Final_Recall@5': metrics['Recall@5']
            })

        # 每個 Reranker 跑完後釋放記憶體
        del model
        cleanup()
        # 增量存檔
        pd.DataFrame(rerank_results).to_csv(os.path.join(OUTPUT_DIR, 'stage2_rerank_results.csv'), index=False)
        
    except Exception as e:
        print(f"Error processing {r_name}: {e}")
        continue

print("\n>>> Benchmark Completed! Check 'ultimate_benchmark_results' folder.")



PHASE 2: Reranking Evaluation

Processing Reranker: [Bi] jinaai/jina-embeddings-v2-base-en


Reranking BM25:   0%|          | 0/41 [00:00<?, ?it/s]


Processing Reranker: [Bi] Alibaba-NLP/gte-large-en-v1.5


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


Processing Reranker: [Late] BAAI/bge-m3


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Reranking BM25:   0%|          | 0/41 [00:00<?, ?it/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████| 13/13 [00:00<00:00, 14.29it/s]

Error processing BAAI/bge-m3: einsum(): the number of subscripts in the equation (2) does not match the number of dimensions (1) for operand 0 and no ellipsis was given

>>> Benchmark Completed! Check 'ultimate_benchmark_results' folder.
